# 01h -- dim_scoring_category Seed

**Purpose**: Build `dim_scoring_category`, the transformer table that puts each Fantrax
Scoring Category in one of seven Category Groups and names the nflverse
`load_player_stats` columns whose sum is its count (ADR-0023, #473 brief 3).
A new category is one SEED row, never code.

`category_code`, `category_name` and the Fantrax block come from the stored
`dim_fantrax_scoring_rule` (04p), so no Fantrax value is typed here. A rules
category with no SEED row, or a SEED row no rules season holds, fails the build
by name. A category with no nflverse column keeps its row with
`nflverse_columns` empty. Reads stored tables only; no network.

**Key**: `category_id` (unique)

**Outputs**:
- `data/dim_scoring_category.parquet`

In [1]:
import sys
from pathlib import Path
for _p in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import CFG, DATA, REVIEW

import pandas as pd

## 1 -- The groups and the SEED

One SEED tuple per Fantrax `category_id`: its Category Group and its
`nflverse_columns`, plus-joined, or `None` where nflverse's season file is not
known to hold the count. The group order is the sort order, 1 to 7.

In [2]:
GROUPS = [
    "Passing",
    "Rushing",
    "Receiving",
    "Special teams",
    "Defense, tackles",
    "Defense, big plays",
    "Other",
]
SPECIAL_TEAMS_GROUP = "Special teams"
SPECIAL_TEAMS_UNIT = "Special Teams"

SEED = [
    ("2760", "Passing", "passing_interceptions"),
    ("2860", "Passing", "passing_tds"),
    ("2870", "Passing", "passing_yards"),
    ("2970", "Rushing", "carries"),
    ("3040", "Rushing", "rushing_tds"),
    ("3050", "Rushing", "rushing_yards"),
    ("2880", "Receiving", "receiving_tds"),
    ("2890", "Receiving", "receiving_yards"),
    ("2900", "Receiving", "receptions"),
    ("3218", "Special teams", "kickoff_return_yards+punt_return_yards"),
    ("256g", "Special teams", None),
    ("3110", "Defense, tackles", "def_tackle_assists"),
    ("3117", "Defense, tackles", "def_tackles_for_loss"),
    ("3120", "Defense, tackles", "def_tackles_solo"),
    ("2680", "Defense, big plays", "def_fumbles_forced"),
    ("2690", "Defense, big plays", "fumble_recovery_opp"),
    ("2710", "Defense, big plays", "fumble_recovery_yards_opp"),
    ("2730", "Defense, big plays", "def_interceptions"),
    ("2750", "Defense, big plays", "def_interception_yards"),
    ("2770", "Defense, big plays", "def_pass_defended"),
    ("3060", "Defense, big plays", "def_sacks"),
    ("3092", "Defense, big plays", "def_safeties"),
    ("3130", "Defense, big plays", "def_tds"),
    ("2670", "Other", "rushing_fumbles_lost+receiving_fumbles_lost+sack_fumbles_lost"),
    ("3147", "Other", "rushing_2pt_conversions+receiving_2pt_conversions"),
    ("2657", "Other", "rushing_first_downs+receiving_first_downs"),
]

COLUMNS = [
    "category_id", "category_code", "category_name", "category_group",
    "group_sort_order", "unit", "nflverse_columns",
]

## 2 -- Build

Each category takes its code, name and Fantrax block from its newest rules
season. `unit` is Special Teams for the Special teams group, else the Fantrax
block (CONTEXT.md, Unit).

In [3]:
def build_dim_scoring_category(rules: pd.DataFrame, seed: list) -> pd.DataFrame:
    seed_df = pd.DataFrame(seed, columns=["category_id", "category_group", "nflverse_columns"])
    repeated = sorted(seed_df.loc[seed_df["category_id"].duplicated(), "category_id"])
    if repeated:
        raise ValueError(f"SEED lists a category_id twice: {repeated}")
    unknown = sorted(set(seed_df["category_group"]) - set(GROUPS))
    if unknown:
        raise ValueError(f"SEED names a group outside the seven: {unknown}")
    cats = (rules.sort_values("season_id", ascending=False, kind="stable")
                 .drop_duplicates("category_id")
                 [["category_id", "category_code", "category_name", "scoring_group"]])
    no_seed = cats[~cats["category_id"].isin(seed_df["category_id"])]
    if not no_seed.empty:
        named = [f"{r.category_id} {r.category_code} ({r.category_name})" for r in no_seed.itertuples()]
        raise ValueError(f"rules categories with no SEED row: {named}")
    no_rule = sorted(set(seed_df["category_id"]) - set(cats["category_id"]))
    if no_rule:
        raise ValueError(f"SEED rows no rules season holds: {no_rule}")
    df = seed_df.merge(cats, on="category_id", how="inner", validate="one_to_one")
    order = {g: i for i, g in enumerate(GROUPS, start=1)}
    df["group_sort_order"] = df["category_group"].map(order).astype("int64")
    df["unit"] = df["scoring_group"].where(df["category_group"] != SPECIAL_TEAMS_GROUP,
                                           SPECIAL_TEAMS_UNIT)
    return (df[COLUMNS].sort_values(["group_sort_order", "category_id"])
                       .reset_index(drop=True))


rules = etl.read_table("dim_fantrax_scoring_rule")
dim_scoring_category = build_dim_scoring_category(rules, SEED)
etl.write_table("dim_scoring_category", dim_scoring_category)
print(f"dim_scoring_category: {len(dim_scoring_category)} rows from "
      f"{rules['category_id'].nunique()} rules categories")

dim_scoring_category: 26 rows from 26 rules categories


## 3 -- Validation

In [4]:
df = etl.read_table("dim_scoring_category")
print(f"dim_scoring_category: {len(df)} rows, {len(df.columns)} columns")
print(f"Duplicate category_id values: {int(df['category_id'].duplicated().sum())}  (expected 0)")
print()
print(df.groupby(["group_sort_order", "category_group"]).size().rename("categories").to_string())
print()
unmapped = df[df["nflverse_columns"].isna()]
print(f"No nflverse column ({len(unmapped)}): "
      f"{[f'{r.category_code} ({r.category_name})' for r in unmapped.itertuples()]}")
print()
print(df.to_string(index=False))

dim_scoring_category: 26 rows, 7 columns
Duplicate category_id values: 0  (expected 0)

group_sort_order  category_group    
1                 Passing               3
2                 Rushing               3
3                 Receiving             3
4                 Special teams         2
5                 Defense, tackles      3
6                 Defense, big plays    9
7                 Other                 3

No nflverse column (1): ['BK (Blocked Kicks)']

category_id category_code                            category_name     category_group  group_sort_order          unit                                              nflverse_columns
       2760           Int                     Interceptions Thrown            Passing                 1       Offense                                         passing_interceptions
       2860            TD                       Passing Touchdowns            Passing                 1       Offense                                                   pass